# Evaluation Metrics for Generative Models

**FID:** distribution distance (lower = better, measures quality + diversity). **CLIP Score:** text-image alignment (higher = better). **LPIPS:** perceptual distance between images (lower = more similar). **FVD:** FID for video. These are imperfect proxies -- know their failure modes. ~2 hrs, needs GPU for generation.

In [ ]:
import torch
import numpy as np
from diffusers import StableDiffusionPipeline
from torchvision import datasets, transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.functional.multimodal import clip_score
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from functools import partial
from PIL import Image
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Generate evaluation set: 200+ images from SD 1.5 with standardized prompts.
# 50 unique prompts x 4 seeds each = 200 images.
# Using fixed seeds for full reproducibility.
# GPU memory: ~3.5 GB for FP16 SD 1.5. Generation takes ~15-20 min.

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to(device)

prompts = [
    "a red car parked on a street",
    "a bowl of fruit on a wooden table",
    "a dog running through a park",
    "a cat sleeping on a windowsill",
    "a mountain landscape at sunset",
    "a city skyline at night",
    "a person reading a book in a library",
    "a boat on a calm lake",
    "a field of sunflowers",
    "a snowy forest path",
    "a cup of coffee on a desk",
    "a bicycle leaning against a wall",
    "a lighthouse on a rocky coast",
    "a hot air balloon over a valley",
    "a child playing with building blocks",
    "a violin on a wooden chair",
    "a train crossing a bridge",
    "a waterfall in a tropical jungle",
    "a plate of sushi on a counter",
    "a deer standing in a meadow",
    "a clock tower in a European town",
    "a surfer riding a wave",
    "a campfire under the stars",
    "a potted plant on a shelf",
    "a piano in an empty room",
    "an old bookshop with wooden shelves",
    "a sailboat at sea during golden hour",
    "a fox in an autumn forest",
    "a rainy street with reflections",
    "a butterfly on a flower",
    "a stone bridge over a stream",
    "a fruit market with colorful stalls",
    "a telescope pointed at the night sky",
    "an eagle soaring above mountains",
    "a cozy cabin in the woods",
    "a glass of wine next to a fireplace",
    "a vintage typewriter on a desk",
    "a coral reef with tropical fish",
    "a windmill in a Dutch landscape",
    "a golden retriever on a beach",
    "a medieval castle on a hill",
    "a beekeeper tending to hives",
    "a row of colorful houses",
    "a misty morning in a rice paddy",
    "a jazz musician playing saxophone",
    "a still life with apples and grapes",
    "a rocket launching into space",
    "a chef cooking in a restaurant kitchen",
    "a hammock between two palm trees",
    "a northern lights display over a frozen lake",
]

print(f"Generating {len(prompts)} prompts x 4 seeds = {len(prompts) * 4} images...")

generated_images = []
for i, p in enumerate(tqdm(prompts, desc="Prompts")):
    for seed in range(4):
        generator = torch.Generator(device=device).manual_seed(i * 4 + seed)
        img = pipe(p, num_inference_steps=20, generator=generator).images[0]
        generated_images.append((img, p))

print(f"Generated {len(generated_images)} images")

In [ ]:
# Compute FID: Frechet Inception Distance
# Measures the distance between the distribution of generated images and real images
# in InceptionV3 feature space. Lower = better.
# Key properties:
#   - Captures both quality AND diversity (mode collapse increases FID)
#   - Needs ~10K+ samples for stable estimates
#   - Sensitive to reference set choice
#   - Using CIFAR-10 as reference (not ideal domain match, but available)

fid = FrechetInceptionDistance(normalize=True).to(device)

# Add real images (CIFAR-10 as reference set)
real_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
    ]),
)
real_loader = torch.utils.data.DataLoader(real_dataset, batch_size=64, shuffle=True)

# Add real distribution (1000 images)
num_real = 0
for batch, _ in tqdm(real_loader, desc="Real images"):
    fid.update(batch.to(device), real=True)
    num_real += batch.shape[0]
    if num_real >= 1000:
        break

# Add generated distribution
for img, _ in tqdm(generated_images, desc="Generated images"):
    img_tensor = transforms.ToTensor()(img.resize((299, 299))).unsqueeze(0).to(device)
    fid.update(img_tensor, real=False)

fid_score = fid.compute()
print(f"\nFID Score: {fid_score:.2f}")
print(f"  (Using {num_real} real images vs {len(generated_images)} generated images)")
print(f"  Note: CIFAR-10 is a poor domain match for SD outputs -- FID will be high.")
print(f"  In practice, use a matched reference set (e.g., COCO for general images).")

del fid
torch.cuda.empty_cache()

In [ ]:
# Compute CLIP scores per image
# Measures how well the generated image matches its text prompt.
# Higher = better alignment. Typical range: 20-35 for SD 1.5.
# Limitation: rewards prompt adherence but ignores visual quality/artifacts.

clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch16")

clip_scores = []
for img, p in tqdm(generated_images[:50], desc="CLIP scores"):
    img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).unsqueeze(0)
    score = clip_score_fn(img_tensor, [p]).item()
    clip_scores.append(score)

print(f"\nCLIP Score stats (n={len(clip_scores)}):")
print(f"  Mean: {np.mean(clip_scores):.2f}")
print(f"  Std:  {np.std(clip_scores):.2f}")
print(f"  Min:  {np.min(clip_scores):.2f}")
print(f"  Max:  {np.max(clip_scores):.2f}")

# Histogram of CLIP scores
plt.figure(figsize=(10, 4))
plt.hist(clip_scores, bins=20, edgecolor="black", alpha=0.7)
plt.xlabel("CLIP Score")
plt.ylabel("Count")
plt.title("Distribution of CLIP Scores (text-image alignment)")
plt.axvline(
    np.mean(clip_scores),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(clip_scores):.1f}",
)
plt.legend()
plt.show()

In [ ]:
# LPIPS: Learned Perceptual Image Patch Similarity
# Measures perceptual distance between two images using deep features.
# Lower = more perceptually similar.
# Here: measure VAE reconstruction quality (encode -> decode should be near-lossless).
# Uses AlexNet features by default (fast, well-calibrated).

lpips = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).to(device)

lpips_scores = []
for img, _ in tqdm(generated_images[:20], desc="LPIPS"):
    # Prepare image: resize to 512x512, normalize to [-1, 1]
    img_tensor = (
        transforms.ToTensor()(img.resize((512, 512))).unsqueeze(0).to(device) * 2 - 1
    )

    # Encode and decode through VAE (round-trip)
    with torch.no_grad():
        latent = pipe.vae.encode(img_tensor.half()).latent_dist.sample()
        reconstructed = pipe.vae.decode(latent).sample.float()

    score = lpips(img_tensor, reconstructed.clamp(-1, 1)).item()
    lpips_scores.append(score)

print(f"\nLPIPS VAE reconstruction stats (n={len(lpips_scores)}):")
print(f"  Mean: {np.mean(lpips_scores):.4f}")
print(f"  Std:  {np.std(lpips_scores):.4f}")
print(f"  (Lower = better reconstruction. Good VAE: < 0.05)")

del lpips
torch.cuda.empty_cache()

In [ ]:
# Show examples: high vs low CLIP scores
# Visually verify that CLIP score correlates with prompt adherence.

scored = sorted(zip(clip_scores, generated_images[:50]), key=lambda x: x[0])

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Bottom 5 (low alignment) vs Top 5 (high alignment) CLIP Scores", fontsize=14)

for i, (score, (img, p)) in enumerate(scored[:5]):
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"CLIP: {score:.1f}\n{p[:30]}...", fontsize=8)
    axes[0, i].axis("off")

for i, (score, (img, p)) in enumerate(scored[-5:]):
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"CLIP: {score:.1f}\n{p[:30]}...", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Low CLIP\n(poor alignment)", fontsize=10)
axes[1, 0].set_ylabel("High CLIP\n(good alignment)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Mini eval pipeline: generate -> compute all metrics -> summary.
# This is what you'd use in practice to compare two model checkpoints.


def evaluate_model(pipe, prompts, n_per_prompt=2, reference_loader=None):
    """Full evaluation pipeline: generate images, compute FID + CLIP + LPIPS."""
    clip_score_fn = partial(clip_score, model_name_or_path="openai/clip-vit-base-patch16")

    # Generate
    images_with_prompts = []
    for p in tqdm(prompts, desc="Generating"):
        for seed in range(n_per_prompt):
            gen = torch.Generator(device=device).manual_seed(seed)
            img = pipe(p, num_inference_steps=20, generator=gen).images[0]
            images_with_prompts.append((img, p))

    # CLIP scores
    cs = []
    for img, p in images_with_prompts:
        t = torch.from_numpy(np.array(img)).permute(2, 0, 1).unsqueeze(0)
        cs.append(clip_score_fn(t, [p]).item())

    # LPIPS (VAE round-trip)
    lpips_metric = LearnedPerceptualImagePatchSimilarity(
        net_type="alex", normalize=True
    ).to(device)
    lp = []
    for img, _ in images_with_prompts[:10]:  # Subset for speed
        img_t = transforms.ToTensor()(img.resize((512, 512))).unsqueeze(0).to(device) * 2 - 1
        with torch.no_grad():
            latent = pipe.vae.encode(img_t.half()).latent_dist.sample()
            recon = pipe.vae.decode(latent).sample.float()
        lp.append(lpips_metric(img_t, recon.clamp(-1, 1)).item())
    del lpips_metric
    torch.cuda.empty_cache()

    return {
        "n_images": len(images_with_prompts),
        "clip_mean": float(np.mean(cs)),
        "clip_std": float(np.std(cs)),
        "clip_min": float(np.min(cs)),
        "clip_max": float(np.max(cs)),
        "lpips_mean": float(np.mean(lp)),
        "lpips_std": float(np.std(lp)),
    }


# Run on a small prompt set
eval_prompts = [
    "a red rose",
    "a mountain landscape",
    "a city at night",
    "a dog playing fetch",
    "a cup of coffee",
]
eval_results = evaluate_model(pipe, eval_prompts)

print("\nEvaluation Results:")
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Why Metrics Aren't Enough

- **FID** needs ~10K+ samples for stability and is sensitive to reference set choice. Different resize methods and feature extractors give different numbers.
- **CLIP score** rewards prompt adherence but ignores visual quality, artifacts, and temporal consistency (for video). A blurry image that matches the prompt can score higher than a sharp one that's slightly off.
- **LPIPS** measures perceptual similarity between two specific images, not generation quality in general.
- **FVD** (Frechet Video Distance) extends FID to video using I3D features but has even more variance and is sensitive to frame sampling.

In practice: metrics gate bad models, but human eval is needed to pick the best. Production labs typically use a mix of automated metrics + internal human eval pipelines.

## Checkpoint

**How to evaluate if a new model checkpoint is better:**

1. **Automated:** compute FID (diversity + quality), CLIP (alignment), LPIPS (perceptual quality) on standard eval set.
2. **Compare** to previous checkpoint's metrics -- look for Pareto improvements (better on one metric without regressing on others).
3. **Human eval:** side-by-side comparisons on curated prompts, focusing on failure modes.
4. **For video:** FVD + temporal consistency metrics + human eval for motion quality.

Never trust a single metric.

**Further reading:** clean-fid ([2104.11222](https://arxiv.org/abs/2104.11222)).